# PathSnap Hackathon — Phase 1 Evaluation Notebook
### PES University — UE24CS352A: Machine Learning

**Objective for Evaluation 1 (11:30 AM):**
1. **Data Preprocessing & Feature Engineering (5 Marks):** Parse `.track`, `.nodes`, `.arcs`, `.route`, metric coordinate projection, candidate arc generation using KD-Tree, spatial & kinematic feature engineering, and interactive Folium EDA.
2. **Model Selection & Training (5 Marks):** Train and compare at least **5 distinct models** from different algorithm families (Logistic Regression, Decision Tree, Random Forest, HistGradientBoosting, KNN) with Group-KFold cross validation, student-test evaluation, and comprehensive metric comparison.

## 1. Setup & Environment

In [ ]:
# If running in Google Colab, install any missing dependencies:
# !pip install folium scikit-learn pandas numpy scipy matplotlib seaborn joblib

import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import joblib
from scipy.spatial import KDTree

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss, precision_score, recall_score, confusion_matrix

print('All libraries loaded successfully!')

## 2. Dataset Paths & Data Loading Utilities
Organized into:
- `.track`: GPS observations (longitude, latitude, timestamp in seconds)
- `.nodes`: Road network nodes (longitude, latitude)
- `.arcs`: Road network segments (from_node_id, to_node_id)
- `.route`: Chronological sequence of ground-truth `arc_id` values

In [ ]:
class TripData:
    def __init__(self, trip_id: str, track: pd.DataFrame, nodes: pd.DataFrame, arcs: pd.DataFrame, route: pd.DataFrame = None):
        self.trip_id = trip_id
        self.track = track  # lon, lat, t
        self.nodes = nodes  # lon, lat
        self.arcs = arcs    # u, v
        self.route = route  # arc_id

    def __repr__(self):
        return f"<TripData {self.trip_id}: {len(self.track)} GPS pts, {len(self.arcs)} arcs, {len(self.route) if self.route is not None else 0} route arcs>"

def load_single_trip(trip_dir: Path) -> TripData:
    trip_id = trip_dir.name
    track = pd.read_csv(trip_dir / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "t"], dtype=float)
    nodes = pd.read_csv(trip_dir / f"{trip_id}.nodes", sep=r"\s+", header=None, names=["lon", "lat"], dtype=float)
    arcs = pd.read_csv(trip_dir / f"{trip_id}.arcs", sep=r"\s+", header=None, names=["u", "v"], dtype=int)
    
    route_path = trip_dir / f"{trip_id}.route"
    route = pd.read_csv(route_path, sep=r"\s+", header=None, names=["arc_id"], dtype=int) if route_path.exists() else None
    
    return TripData(trip_id=trip_id, track=track, nodes=nodes, arcs=arcs, route=route)

# Set base directories (adjust path if running in Colab)
DATA_DIR = Path("Student folder") if Path("Student folder").exists() else Path(".")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "student_test"

sample_trip = load_single_trip(TRAIN_DIR / "00000001")
print("Sample trip loaded:", sample_trip)

## 3. Exploratory Data Analysis (EDA) & Folium Visualization
Rubric check: missing values, sampling intervals (~97% at 1Hz), and interactive map.

In [ ]:
# Check missing values and sampling intervals
missing_vals = sample_trip.track.isna().sum().to_dict()
dt = np.diff(sample_trip.track["t"].values)
exact_1s = (np.abs(dt - 1.0) < 1e-3).sum()
pct_1s = (exact_1s / len(dt)) * 100.0

print("Missing GPS values:", missing_vals)
print(f"Mean sampling interval: {np.mean(dt):.3f}s, exactly 1s apart: {pct_1s:.2f}%")

# Generate Folium Map for Sample Trip
mean_lat = sample_trip.track["lat"].mean()
mean_lon = sample_trip.track["lon"].mean()
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=15, tiles="OpenStreetMap")

# Ground Truth Route (Green)
if sample_trip.route is not None:
    route_arc_ids = sample_trip.route["arc_id"].values
    route_lines = []
    for aid in route_arc_ids:
        u = sample_trip.arcs.iloc[aid]["u"]
        v = sample_trip.arcs.iloc[aid]["v"]
        route_lines.append([
            [sample_trip.nodes.iloc[u]["lat"], sample_trip.nodes.iloc[u]["lon"]],
            [sample_trip.nodes.iloc[v]["lat"], sample_trip.nodes.iloc[v]["lon"]]
        ])
    folium.PolyLine(locations=route_lines, color="#2ecc71", weight=5, opacity=0.8, tooltip="Ground Truth Route").add_to(m)

# Raw GPS Path (Blue)
gps_coords = sample_trip.track[["lat", "lon"]].values.tolist()
folium.PolyLine(locations=gps_coords, color="#3498db", weight=2, dash_array="5, 5", tooltip="Raw GPS Path").add_to(m)
for lat, lon in gps_coords[::10]:
    folium.CircleMarker(location=[lat, lon], radius=3, color="#2980b9", fill=True, fill_color="#3498db", fill_opacity=0.9).add_to(m)

os.makedirs("artifacts/maps", exist_ok=True)
m.save("artifacts/maps/sample_trip_eda.html")
print("Map saved to artifacts/maps/sample_trip_eda.html")
m

## 4. Geometric Projections & Spatial Feature Engineering
We calculate:
- Metric projected coordinates $(x, y)$ in meters (equirectangular centered at mean trajectory latitude)
- Shortest perpendicular/orthogonal distance from GPS point to candidate road segment
- Road segment bearing and vehicle movement heading
- Angular heading difference ($[0, 180]$ degrees)
- Distance to start/end nodes & projection ratio $t \in [0, 1]$
- KD-Tree retrieval of top-$K$ nearest candidate arcs per observation

In [ ]:
EARTH_RADIUS = 6371008.8

def to_metric_coords(lons, lats, ref_lon, ref_lat):
    rad_lat = np.radians(ref_lat)
    x = np.radians(lons - ref_lon) * EARTH_RADIUS * np.cos(rad_lat)
    y = np.radians(lats - ref_lat) * EARTH_RADIUS
    return x, y

def angle_difference(a, b):
    diff = abs(a - b) % 360.0
    return diff if diff <= 180.0 else 360.0 - diff

def point_to_segment_proj(px, py, ax, ay, bx, by):
    dx, dy = bx - ax, by - ay
    l2 = dx*dx + dy*dy
    if l2 == 0.0:
        return np.hypot(px - ax, py - ay), 0.0
    t = max(0.0, min(1.0, ((px - ax)*dx + (py - ay)*dy) / l2))
    proj_x = ax + t * dx
    proj_y = ay + t * dy
    return np.hypot(px - proj_x, py - proj_y), t

def align_route_arcs(trip, gps_x, gps_y, node_x, node_y):
    if trip.route is None:
        return np.full(len(gps_x), -1, dtype=int)
    route_arcs = trip.route["arc_id"].values
    r_u = trip.arcs.loc[route_arcs, "u"].values
    r_v = trip.arcs.loc[route_arcs, "v"].values
    r_ax, r_ay = node_x[r_u], node_y[r_u]
    r_bx, r_by = node_x[r_v], node_y[r_v]
    
    true_arcs = np.zeros(len(gps_x), dtype=int)
    cur_idx = 0
    for i in range(len(gps_x)):
        px, py = gps_x[i], gps_y[i]
        best_d = 1e9
        best_r = cur_idx
        for r in range(cur_idx, min(len(route_arcs), cur_idx + 20)):
            d, _ = point_to_segment_proj(px, py, r_ax[r], r_ay[r], r_bx[r], r_by[r])
            if d < best_d:
                best_d = d
                best_r = r
        cur_idx = best_r
        true_arcs[i] = route_arcs[best_r]
    return true_arcs

def extract_features_from_trip(trip: TripData, k_candidates: int = 10, is_training: bool = True) -> pd.DataFrame:
    ref_lon, ref_lat = trip.track["lon"].mean(), trip.track["lat"].mean()
    node_x, node_y = to_metric_coords(trip.nodes["lon"].values, trip.nodes["lat"].values, ref_lon, ref_lat)
    gps_x, gps_y = to_metric_coords(trip.track["lon"].values, trip.track["lat"].values, ref_lon, ref_lat)
    
    u_idx, v_idx = trip.arcs["u"].values, trip.arcs["v"].values
    ax, ay = node_x[u_idx], node_y[u_idx]
    bx, by = node_x[v_idx], node_y[v_idx]
    
    seg_lengths = np.hypot(bx - ax, by - ay)
    seg_bearings = (np.degrees(np.arctan2(bx - ax, by - ay)) + 360.0) % 360.0
    tree = KDTree(np.column_stack([(ax + bx) / 2.0, (ay + by) / 2.0]))
    
    times = trip.track["t"].values
    dt = np.diff(times, prepend=times[0] - 1.0)
    dt[dt <= 0] = 1.0
    gps_dx = np.diff(gps_x, prepend=gps_x[0])
    gps_dy = np.diff(gps_y, prepend=gps_y[0])
    speeds = np.hypot(gps_dx, gps_dy) / dt
    headings = (np.degrees(np.arctan2(gps_dx, gps_dy)) + 360.0) % 360.0
    
    true_arcs = align_route_arcs(trip, gps_x, gps_y, node_x, node_y) if is_training else np.full(len(gps_x), -1)
    _, cand_indices = tree.query(np.column_stack([gps_x, gps_y]), k=k_candidates)
    
    records = []
    for i in range(len(gps_x)):
        px, py = gps_x[i], gps_y[i]
        target = true_arcs[i]
        candidates = cand_indices[i]
        
        cand_dists = [point_to_segment_proj(px, py, ax[c], ay[c], bx[c], by[c])[0] for c in candidates]
        sorted_idx = np.argsort(cand_dists)
        
        for rank, s_idx in enumerate(sorted_idx):
            c = candidates[s_idx]
            d, t = point_to_segment_proj(px, py, ax[c], ay[c], bx[c], by[c])
            head_diff = angle_difference(headings[i], seg_bearings[c])
            label = 1 if (is_training and c == target) else 0
            
            records.append({
                "trip_id": trip.trip_id,
                "obs_id": i,
                "candidate_arc_id": int(c),
                "dist_to_segment": float(d),
                "projection_ratio": float(t),
                "heading_diff": float(head_diff),
                "segment_length": float(seg_lengths[c]),
                "dist_to_start": float(np.hypot(px - ax[c], py - ay[c])),
                "dist_to_end": float(np.hypot(px - bx[c], py - by[c])),
                "vehicle_speed": float(speeds[i]),
                "candidate_rank": int(rank),
                "target_arc_id": int(target),
                "is_match": int(label)
            })
    return pd.DataFrame(records)

print('Feature extraction pipeline defined!')

## 5. Build Training & Testing Datasets
Extract candidate feature vectors from training and student-testing partitions.

In [ ]:
def prepare_partition(folder: Path, max_trips: int = 15):
    subfolders = sorted([f for f in folder.iterdir() if f.is_dir() and not f.name.startswith(".")])[:max_trips]
    print(f"Processing {len(subfolders)} trips from {folder.name}...")
    dfs = [extract_features_from_trip(load_single_trip(f), k_candidates=10, is_training=True) for f in subfolders]
    return pd.concat(dfs, ignore_index=True)

train_df = prepare_partition(TRAIN_DIR, max_trips=15)
test_df = prepare_partition(TEST_DIR, max_trips=10)

feature_cols = [
    "dist_to_segment",
    "projection_ratio",
    "heading_diff",
    "segment_length",
    "dist_to_start",
    "dist_to_end",
    "vehicle_speed",
    "candidate_rank"
]

X_train = train_df[feature_cols].values
y_train = train_df["is_match"].values
groups_train = train_df["trip_id"].values

X_test = test_df[feature_cols].values
y_test = test_df["is_match"].values

print(f"Training set: {X_train.shape} across {len(np.unique(groups_train))} trips")
print(f"Student-test set: {X_test.shape} across {len(np.unique(test_df['trip_id']))} trips")

## 6. Train & Evaluate 5 Distinct Machine Learning Models
Mandated algorithm families:
1. **Logistic Regression** (Linear & Probabilistic)
2. **K-Nearest Neighbors** (Instance-based)
3. **Decision Tree** (Tree-based)
4. **Random Forest** (Ensemble)
5. **HistGradientBoosting** (Boosting)

Evaluated using:
- **Group-KFold Cross-Validation** (prevents data leakage across trips)
- **Observation-level Top-1 Accuracy** (highest scored candidate per GPS point)
- **Macro F1-Score & Precision/Recall**
- **Log Loss**
- **Inference & Training Times**

In [ ]:
def eval_top1_accuracy(df_eval, probs):
    df_temp = df_eval[["trip_id", "obs_id", "is_match"]].copy()
    df_temp["score"] = probs
    idx_top1 = df_temp.groupby(["trip_id", "obs_id"])["score"].idxmax()
    return float(np.mean(df_temp.loc[idx_top1, "is_match"].values))

models = {
    "Logistic Regression": {
        "family": "Linear & Probabilistic",
        "config": "C=1.0, solver='lbfgs', max_iter=500",
        "pipe": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(C=1.0, max_iter=500, random_state=42))])
    },
    "Decision Tree": {
        "family": "Tree-based",
        "config": "max_depth=10, min_samples_split=20",
        "pipe": Pipeline([("scaler", StandardScaler()), ("clf", DecisionTreeClassifier(max_depth=10, min_samples_split=20, random_state=42))])
    },
    "Random Forest": {
        "family": "Ensemble",
        "config": "n_estimators=50, max_depth=12, n_jobs=-1",
        "pipe": Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1))])
    },
    "HistGradientBoosting": {
        "family": "Boosting",
        "config": "max_iter=100, learning_rate=0.1, max_depth=6",
        "pipe": Pipeline([("scaler", StandardScaler()), ("clf", HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=6, random_state=42))])
    },
    "K-Nearest Neighbors": {
        "family": "Instance-based",
        "config": "n_neighbors=7, weights='distance'",
        "pipe": Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=7, weights="distance"))])
    }
}

# 1. Group-aware Cross-Validation
gkf = GroupKFold(n_splits=3)
cv_results = {}
print("--- 1. Group-aware Cross Validation ---")
for name, m in models.items():
    pipe = m["pipe"]
    sub_step = 5 if name == "K-Nearest Neighbors" else 1
    idx_sub = np.arange(0, len(X_train), sub_step)
    X_sub, y_sub, g_sub = X_train[idx_sub], y_train[idx_sub], groups_train[idx_sub]
    
    f1_list = []
    for tr, val in gkf.split(X_sub, y_sub, groups=g_sub):
        pipe.fit(X_sub[tr], y_sub[tr])
        preds = pipe.predict(X_sub[val])
        f1_list.append(f1_score(y_sub[val], preds, average="macro"))
    cv_results[name] = float(np.mean(f1_list))
    print(f"[{name}] Group CV Macro F1: {cv_results[name]:.4f}")

# 2. Full Train and Test Evaluation
comparison_table = []
trained_pipelines = {}
print("\n--- 2. Training on Train and Evaluating on Student-Test ---")
for name, m in models.items():
    pipe = m["pipe"]
    
    t0 = time.time()
    if name == "K-Nearest Neighbors":
        sub_idx = np.arange(0, len(X_train), 4)
        pipe.fit(X_train[sub_idx], y_train[sub_idx])
    else:
        pipe.fit(X_train, y_train)
    train_t = time.time() - t0
    
    t1 = time.time()
    probs = pipe.predict_proba(X_test)[:, 1]
    inf_t = time.time() - t1
    preds = (probs >= 0.5).astype(int)
    
    top1 = eval_top1_accuracy(test_df, probs)
    test_f1 = f1_score(y_test, preds, average="macro")
    loss = log_loss(y_test, np.clip(probs, 1e-6, 1 - 1e-6))
    
    trained_pipelines[name] = pipe
    comparison_table.append({
        "Model": name,
        "Algorithm Family": m["family"],
        "Key Configuration": m["config"],
        "CV Macro F1": round(cv_results[name], 4),
        "Top-1 Accuracy": round(top1, 4),
        "Student-Test Macro F1": round(test_f1, 4),
        "Log Loss": round(loss, 4),
        "Inference Time (s)": round(inf_t, 4),
        "Training Time (s)": round(train_t, 4)
    })
    print(f"[{name}] Top-1 Acc: {top1*100:.2f}% | Test Macro F1: {test_f1:.4f} | Log Loss: {loss:.4f}")

df_comparison = pd.DataFrame(comparison_table)
os.makedirs("artifacts/results", exist_ok=True)
df_comparison.to_csv("artifacts/results/model_comparison.csv", index=False)
print("\n=== Model Comparison Exported to artifacts/results/model_comparison.csv ===")
df_comparison

## 7. Visualizations & Performance Comparison Charts
Compulsory charts for the technical report:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Macro F1 comparison
sns.barplot(data=df_comparison, x="Model", y="Student-Test Macro F1", ax=axes[0], palette="viridis")
axes[0].set_title("Student-Test Macro F1-Score")
axes[0].set_ylim(0.5, 1.0)
axes[0].tick_params(axis='x', rotation=30)

# 2. Top-1 Observation Accuracy
sns.barplot(data=df_comparison, x="Model", y="Top-1 Accuracy", ax=axes[1], palette="magma")
axes[1].set_title("Observation-Level Top-1 Accuracy")
axes[1].set_ylim(0.5, 1.0)
axes[1].tick_params(axis='x', rotation=30)

# 3. Log Loss
sns.barplot(data=df_comparison, x="Model", y="Log Loss", ax=axes[2], palette="rocket")
axes[2].set_title("Log Loss (Lower is better)")
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("artifacts/results/model_comparison_plots.png", dpi=300)
plt.show()

## 8. Export Trained Model Artifacts
Exports best-performing trained model weights for Phase 2 deployment.

In [ ]:
os.makedirs("artifacts/models", exist_ok=True)
for name, pipe in trained_pipelines.items():
    clean_name = name.lower().replace(' ', '_').replace('-', '_')
    joblib.dump(pipe, f"artifacts/models/{clean_name}.joblib")
print("All trained model artifacts saved in artifacts/models/")